# MCP Transport

**Prerequisites:** all of `01`–`04`. Helpful: `01_foundations/08_asyncio.ipynb`

Transport is usually presented as a deployment detail — stdio for local servers, HTTP for remote ones, pick one. That framing is not wrong and it buries the only part that matters.

Notebook 1 said the difference between a local tool and an MCP tool is that one is imported and the other is *launched*, and treated the process boundary as the thing that makes MCP interesting. This notebook is about a fact that framing hides:

**The process boundary and the trust boundary are not the same line.**

A stdio server is a separate process, which sounds like containment and is not. We start it, so it begins life holding whatever we were holding. The first half of this notebook measures that. The second half deals with the ordinary mechanics — lifetimes, timeouts, HTTP, and the Windows failure that nearly stopped this chapter.

## Setup

`transport_server.py` is an `expose()`d registry like notebook 4's, but its tools are instruments rather than tools — they report on the process the server is running in.

In [1]:
import os
import subprocess
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

import psutil

from client import can_be_inherited
from session import Session

TRANSPORT_SERVER = Path.cwd().parent / "transport_server.py"

# Planted on purpose, and not a real key. Everything below asks about this one by name; the
# notebook never lists the machine's actual variables, because its outputs are committed.
os.environ["FORGE_FAKE_API_KEY"] = "sk-not-a-real-key-0123456789"

def ask(session, tool, **args):
    return session.call(tool, args).content[0].text

## A subprocess is not a sandbox

Start a server the way every notebook in this chapter has, and ask it what it received.

In [2]:
with Session(TRANSPORT_SERVER) as s:
    print(ask(s, "whoami"))
    print(ask(s, "env_count"))
    print(ask(s, "env_names", prefix="FORGE_"))
    print(ask(s, "can_you_see", name="FORGE_FAKE_API_KEY"))

pid=21556 cwd=C:\Users\91879\OneDrive\AI Projects (Venkat)\agentic-forge-cookbook\07_mcp
97 variables inherited
1 variable(s) starting with 'FORGE_': FORGE_FAKE_API_KEY
FORGE_FAKE_API_KEY: present, 28 characters


A different process, a different pid — and our API key.

Nothing exotic happened. A child process inherits its parent's environment, that is how processes work, and `FORGE_FAKE_API_KEY` is standing in for every real credential the notebook kernel happens to be holding: the Groq key chapter 1 uses, the Gemini key `04_memory` needs for embeddings, the Neo4j password.

So the server can read them. It can also open files as us, reach the network as us, and write anywhere we can write. **Its separateness bought isolation of failures, not isolation of authority** — notebook 1's claim that a crash in the server does not take down the agent is still true, and is a much smaller claim than it sounds.

## The SDK was careful. We undid it.

The interesting part is whose fault this is, and it is not the protocol's.

In [3]:
from mcp.client.stdio import DEFAULT_INHERITED_ENV_VARS

print(f"what stdio_client inherits by default ({len(DEFAULT_INHERITED_ENV_VARS)} variables):")
print(" ", ", ".join(DEFAULT_INHERITED_ENV_VARS))

what stdio_client inherits by default (12 variables):
  APPDATA, HOMEDRIVE, HOMEPATH, LOCALAPPDATA, PATH, PATHEXT, PROCESSOR_ARCHITECTURE, SYSTEMDRIVE, SYSTEMROOT, TEMP, USERNAME, USERPROFILE


An allowlist. `PATH`, `TEMP`, `USERPROFILE` and similar — the things a process needs in order to start — and not one credential among them. Left alone, `stdio_client` gives a server an environment with no secrets in it at all, and the SDK builds the child's environment as:

```python
env = get_default_environment() | (server.env or {})
```

Which is where it goes wrong. Whatever you put in `StdioServerParameters.env` is merged **on top of** that allowlist, and the obvious thing to write is the whole environment:

```python
env={**os.environ, **(env or {})}     # client.py, and a great many real MCP configs
```

That is our own [`stdio_params`](../client.py), and it overrides a careful default with everything. The protocol did not hand the server our keys. We did, in one line that looks like housekeeping.

## Handing over less

In [4]:
rows = [
    ("inherit=True  (the default)", {}),
    ("inherit=False", {"inherit": False}),
    ("inherit=False, one named", {"inherit": False, "env": {"FORGE_FAKE_API_KEY": "only-this"}}),
]

for label, kwargs in rows:
    with Session(TRANSPORT_SERVER, **kwargs) as s:
        count = ask(s, "env_count")
        key = ask(s, "can_you_see", name="FORGE_FAKE_API_KEY")
    print(f"{label:30} {count:24} {key}")

inherit=True  (the default)    97 variables inherited   FORGE_FAKE_API_KEY: present, 28 characters


inherit=False                  18 variables inherited   FORGE_FAKE_API_KEY: not set in this process


inherit=False, one named       19 variables inherited   FORGE_FAKE_API_KEY: present, 9 characters


**Everything down to 18 variables**, and the key is gone — then back at 9 characters instead of 28, because the third row named it explicitly and gave it a different value.

That last row is the shape to aim for, and it is the same idea `expose()` used in notebook 4 for tools: **nothing crosses implicitly.** A server that needs a key can be given that key; it does not need the others, and the difference between those two sentences is the entire distance between a server you run and a server you trust.

`inherit=True` stays the default in `client.py` on purpose. It is what the ecosystem does, and a chapter that quietly did otherwise would misrepresent what running a stdio server normally costs. `08_guardrails` picks this up — the argument already exists, one layer down.

## Who owns the process

The other thing stdio decides, and it does not look like a transport question either.

In [5]:
s = Session(TRANSPORT_SERVER)
s.__enter__()

pid = int(ask(s, "whoami").split("pid=")[1].split()[0])
print(f"child pid {pid}: running = {psutil.pid_exists(pid)}")

s.__exit__(None, None, None)
time.sleep(0.5)
print(f"after closing:   running = {psutil.pid_exists(pid)}")

child pid 23424: running = True


after closing:   running = False


We started it, so we are responsible for stopping it — and **closing the client's stdin is the only way to say so.** There is no shutdown message in the protocol, no port to close, no way for the server to be told anything after the pipe is gone.

Which means a session that is never closed is a process that never ends. Notebook 1 launched one per cell; a long-running agent that reconnects on every request would leak one per request. The demo server is a few kilobytes and does nothing, but a real one might be holding a database connection or a headless browser.

## Two clocks

Notebook 4 ended on this. `expose(timeout=3)` is what the server allows a tool; `Session(timeout=...)` is what the client waits. They are set by different people, in different files, and neither knows about the other.

In [6]:
with Session(TRANSPORT_SERVER, timeout=30) as s:
    started = time.perf_counter()
    r = s.call("slow", {"seconds": 5})
    elapsed = time.perf_counter() - started
    kind = (r.meta or {}).get("forge/kind")

print(f"server allows 3s, client waits 30s, tool wants 5s")
print(f"   gave up after {elapsed:.1f}s, kind={kind}")
print(f"   {r.content[0].text[:78]}")

server allows 3s, client waits 30s, tool wants 5s
   gave up after 3.0s, kind=timeout
   'slow' took longer than 3 seconds and was given up on. It only reads, so calli


The server's limit won, and because it won *inside* `invoke()`, the failure is a proper `timeout` — `forge/kind` says so, `safe_to_retry` can be computed from it, and the message says whether calling again is safe.

Now put the client's clock in front of the server's.

In [7]:
s = Session(TRANSPORT_SERVER, timeout=30, startup=30)
s.__enter__()
pid = int(ask(s, "whoami").split("pid=")[1].split()[0])

started = time.perf_counter()
try:
    s.call("slow", {"seconds": 5}, timeout=1)
except Exception as e:
    print(f"client waits 1s, server allows 3s, tool wants 5s")
    print(f"   raised {type(e).__name__} after {time.perf_counter() - started:.1f}s")

print(f"   server process still alive: {psutil.pid_exists(pid)}")
print(f"   and still answering:        {ask(s, 'whoami')}")

s.__exit__(None, None, None)
print("   session closed")

client waits 1s, server allows 3s, tool wants 5s
   raised TimeoutError after 1.0s
   server process still alive: True
   and still answering:        pid=23928 cwd=C:\Users\91879\OneDrive\AI Projects (Venkat)\agentic-forge-cookbook\07_mcp


   session closed


Two things worth separating there.

The client gave up and got an exception rather than a `Result`. Notebook 2 mapped that to `broken`, which is the right call — a call that did not come back is not something the model can act on — but note that no `forge/kind` was involved, because nothing on the server ever decided anything. The server was still busy.

And **giving up is not cancelling.** The client stopped waiting; the tool kept running to its full five seconds. There is no message in MCP that means "stop doing that", so a client-side timeout leaves work in flight, finishing into a void. That matters most for the tools where it matters most: `03_tools` made `repeatable` a property of a tool because after a timeout nobody knows whether it went through, and this is why — the timeout is a statement about the *caller's patience*, not about the work.

The rule that falls out: **the server's limit should be the tighter one.** It is the only one attached to the thing actually doing the work.

## The same server, over HTTP

`transport_server.py` takes an argument. Nothing above that line changes.

In [8]:
PORT, URL = 8765, "http://127.0.0.1:8765/mcp"

# Started with a clean environment, standing in for a server somebody else is running
# somewhere else -- which is the normal case for HTTP, and the point of the comparison.
clean = {k: v for k, v in os.environ.items() if not k.startswith("FORGE_")}
listener = subprocess.Popen(
    [sys.executable, str(TRANSPORT_SERVER), "http", str(PORT)],
    cwd=str(TRANSPORT_SERVER.parent), env=clean,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(4)

try:
    with Session(url=URL, timeout=20) as s:
        print("connected to:", s.server_name)
        print("tools:       ", [t.name for t in s.list_tools()])
        print(" ", ask(s, "whoami"))
        print(" ", ask(s, "can_you_see", name="FORGE_FAKE_API_KEY"))
finally:
    listener.terminate()
    listener.wait(timeout=10)
print("listener stopped")

connected to: forge-transport
tools:        ['whoami', 'env_count', 'can_you_see', 'env_names', 'slow']
  pid=9516 cwd=C:\Users\91879\OneDrive\AI Projects (Venkat)\agentic-forge-cookbook\07_mcp
  FORGE_FAKE_API_KEY: not set in this process
listener stopped


Same tools, same schemas, same `Result` mapping — and no key, because this one was not started by us with our environment.

Be careful about the conclusion, because the obvious one is wrong. **HTTP is not safer. Being started by someone else is.** An HTTP server you launch yourself, inheriting your environment, has exactly the problem stdio has. The correlation is real but it is a correlation: stdio servers are almost always launched by their client, HTTP servers almost never are.

What genuinely changes with HTTP is who holds each concern:

| | stdio | streamable HTTP |
|---|---|---|
| who starts it | the client, every time | somebody, once, elsewhere |
| what it inherits | your environment, unless you say otherwise | nothing of yours |
| lifetime | exactly one session | outlives every client |
| how it stops | you close stdin | you do not |
| who else can reach it | nobody — there is no address | anyone who can route to the URL |
| authentication | meaningless, it is your own child | **the whole problem** |
| what a crash costs | that one session | every client at once |

The last two rows are the trade. stdio needs no authentication because the server has no independent existence; HTTP needs it because the server is a service, and a service with no authentication is a tool anyone on the network may call. MCP has OAuth support for exactly this, and it is out of scope here — `08_guardrails` is where it belongs.

## The Windows failure that nearly stopped this chapter

One transport detail earned its own place, because it is the kind that eats an afternoon.

In [9]:
print("can this notebook's stderr be handed to a child process?", can_be_inherited(sys.stderr))
print("type of sys.stderr:", type(sys.stderr).__name__)

can this notebook's stderr be handed to a child process? False
type of sys.stderr: OutStream


`False`, and the reason is three layers down from anything you would think to check.

ipykernel runs a `SelectorEventLoop` on Windows, because pyzmq requires one. Selector loops have no async subprocess support, so `mcp` catches the `NotImplementedError` and falls back to a plain `subprocess.Popen`. `Popen` has to give the child a real OS handle for its stderr. And in a kernel, `sys.stderr` is an ipykernel `OutStream` that forwards over a socket — it answers `write` and raises on `fileno`.

The traceback says `io.UnsupportedOperation: fileno` and mentions neither Jupyter nor stdio.

The fix is not about async at all: give the child a real file. [`connect()`](../client.py) checks whether stderr can be inherited and falls back to `server.log`, which is worth having regardless — a stdio server's stderr is the only place its crashes go, and every `raises` call in notebook 1 landed there.

In [10]:
from client import SERVER_LOG

print("server.log exists:", SERVER_LOG.exists())
if SERVER_LOG.exists():
    print("size:", SERVER_LOG.stat().st_size, "bytes")
    print("it is gitignored, and it is the first thing to open when a server misbehaves")

server.log exists: True
size: 200341 bytes
it is gitignored, and it is the first thing to open when a server misbehaves


## One piece of folklore, tested

The standard advice for Windows stdio servers is to force UTF-8 on the child with `PYTHONIOENCODING`. Under mcp 2.x it does nothing.

In [11]:
TEXT = "café — em dash ✓ 日本"
DEMO = Path.cwd().parent / "server.py"      # the notebook 1 server; it has an `echo` tool

for label, encoding in [("utf-8", "utf-8"), ("cp1252", "cp1252"), ("ascii", "ascii")]:
    with Session(DEMO, env={"PYTHONIOENCODING": encoding}) as s:
        back = s.call("echo", {"text": TEXT}).content[0].text
    print(f"child PYTHONIOENCODING={label:7} -> {back!r}  identical: {back == TEXT}")

child PYTHONIOENCODING=utf-8   -> 'café — em dash ✓ 日本'  identical: True


child PYTHONIOENCODING=cp1252  -> 'café — em dash ✓ 日本'  identical: True


child PYTHONIOENCODING=ascii   -> 'café — em dash ✓ 日本'  identical: True


Round-tripped byte-identical through a child forced to **ascii**. The variable arrives and changes nothing: mcp 2.x reads and writes the pipe as **bytes** and frames UTF-8 itself, so the console code page is never consulted. The advice is real, and it is about mcp 1.x, where the server wrapped `sys.stdin` and `sys.stdout` as text streams and inherited the code page from the console.

Worth knowing mainly because of the failure it used to cause — one em dash in a tool result killing the whole connection rather than the one call — which is the kind of thing that gets blamed on the protocol for years after it stops being true.

## Where this leaves the chapter

The mechanics are small: one transport launches a process and owns it, the other talks to something already running. Almost nothing in `session.py` changes between them, which is the honest measure of how much transport affects your code.

What it affects is everything around your code:

- a stdio server **starts with your authority**, not merely in your process tree. That is a default, and `inherit=False` changes it
- you own its lifetime, and closing stdin is the only way to end it
- a timeout is a statement about the caller's patience; nothing cancels the work, so the server's limit should be the tighter one
- HTTP moves the server out of your control, which removes the inheritance problem and creates an authentication one

`06_build_an_mcp_agent.ipynb` puts all of it inside chapter 2's loop. Then `08_guardrails` takes the first bullet seriously, because "the tool runs in another process" turns out to be a much weaker safety property than it sounds, and this notebook is where that stops being a theoretical objection.